In [1]:
# Cellule 1 — 📦 Installation des dépendances
# ==========================================
# On utilise le kernel Azure ML "Python 3.10 - SDK v2"
# aiohttp et beautifulsoup4 sont déjà installés
# On installe uniquement openai si nécessaire

import subprocess
import sys

packages = ["openai", "aiohttp", "beautifulsoup4", "lxml"]

for pkg in packages:
    try:
        __import__(pkg if pkg != "beautifulsoup4" else "bs4")
        print(f"✅ {pkg} — déjà installé")
    except ImportError:
        print(f"📦 Installation de {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])
        print(f"✅ {pkg} — installé")

print("\n✅ Toutes les dépendances sont prêtes")

✅ openai — déjà installé
✅ aiohttp — déjà installé
✅ beautifulsoup4 — déjà installé
✅ lxml — déjà installé

✅ Toutes les dépendances sont prêtes


In [2]:
# Cellule 2 — 📚 Imports
# ==========================================
import os
import json
import asyncio
import aiohttp
import hashlib
import logging
import time
import re
from datetime import datetime, timedelta
from bs4 import BeautifulSoup
from urllib.parse import urlparse
from typing import Optional, List, Dict
from openai import AzureOpenAI

# Configuration du logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s"
)
logger = logging.getLogger(__name__)

print("✅ Imports chargés")

✅ Imports chargés


In [4]:
# Cellule 3 — 🔑 Connexion Azure OpenAI
# ==========================================
# ⚠️ MODIFIER ICI avec tes informations Azure OpenAI
#
# Pour trouver ces infos :
#   1. Azure Portal → Azure OpenAI → ta ressource
#   2. Onglet "Keys and Endpoint"
#   3. Copie l'endpoint et la clé

AZURE_OPENAI_CONFIG = {
    "endpoint": "https://oai-subul-tuteur-dev.openai.azure.com/", 
    "api_key": "AtUsLTTtFlNA1cBoSprUS5UNX50PUDtHDJ7orkCrCsjoULxkiX2PJQQJ99CBACHYHv6XJ3w3AAABACOGagoi",                          
    "api_version": "2024-12-01-preview",
    "model": "gpt-4o-mini",                                
}

# Initialiser le client
client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_CONFIG["endpoint"],
    api_key=AZURE_OPENAI_CONFIG["api_key"],
    api_version=AZURE_OPENAI_CONFIG["api_version"],
)

# Test de connexion
try:
    test = client.chat.completions.create(
        model=AZURE_OPENAI_CONFIG["model"],
        messages=[{"role": "user", "content": "Réponds juste OK"}],
        max_tokens=5,
    )
    print(f"✅ Azure OpenAI connecté : {test.choices[0].message.content}")
    print(f"   Modèle : {AZURE_OPENAI_CONFIG['model']}")
    print(f"   Endpoint : {AZURE_OPENAI_CONFIG['endpoint']}")
except Exception as e:
    print(f"❌ Erreur connexion : {e}")
    print("   → Vérifie ton endpoint et ta clé API dans Azure Portal")

✅ Azure OpenAI connecté : OK
   Modèle : gpt-4o-mini
   Endpoint : https://oai-subul-tuteur-dev.openai.azure.com/


In [5]:
# Cellule 4 — 📁 Configuration des chemins
# ==========================================
BASE_DIR = os.getcwd()
DATA_DIR = os.path.join(BASE_DIR, "data")

PROVIDERS = ["aws", "azure", "gcp"]

# Créer tous les dossiers
for provider in PROVIDERS:
    os.makedirs(os.path.join(DATA_DIR, "raw", provider), exist_ok=True)
    os.makedirs(os.path.join(DATA_DIR, "processed", provider), exist_ok=True)
    os.makedirs(os.path.join(DATA_DIR, "vector_store", provider), exist_ok=True)

os.makedirs(os.path.join(DATA_DIR, "logs"), exist_ok=True)

print(f"📍 BASE_DIR : {BASE_DIR}")
print(f"📍 DATA_DIR : {DATA_DIR}")
print("✅ Dossiers vérifiés")

📍 BASE_DIR : /mnt/batch/tasks/shared/LS_root/mounts/clusters/dev-station-skander/code/Users/Skander.Kammoun/Agent_Tuteur_Subul/01_Knowledge_Base
📍 DATA_DIR : /mnt/batch/tasks/shared/LS_root/mounts/clusters/dev-station-skander/code/Users/Skander.Kammoun/Agent_Tuteur_Subul/01_Knowledge_Base/data
✅ Dossiers vérifiés


In [6]:
# Cellule 5 — 📚 Liste des cours à générer
# ==========================================
# Chaque cours contient :
#   - topic          : Nom du service cloud
#   - category       : Catégorie (Compute, Storage, Database, etc.)
#   - provider       : aws, azure ou gcp
#   - urls           : URLs officielles à scraper
#   - search_queries : Requêtes pour enrichir le cours via GPT

COURSES = [
    # ═════════════════════════════════════════════
    #  ☁️  AWS — Amazon Web Services
    # ═════════════════════════════════════════════

    # 🖥️ Compute
    {
        "topic": "Amazon EC2",
        "category": "Compute",
        "provider": "aws",
        "urls": [
            "https://docs.aws.amazon.com/AWSEC2/latest/UserGuide/concepts.html",
        ],
        "search_queries": [
            "AWS EC2 instance types explained",
            "AWS EC2 best practices security",
        ],
    },
    {
        "topic": "AWS Lambda",
        "category": "Compute",
        "provider": "aws",
        "urls": [
            "https://docs.aws.amazon.com/lambda/latest/dg/welcome.html",
        ],
        "search_queries": [
            "AWS Lambda serverless complete guide",
            "AWS Lambda best practices",
        ],
    },
    {
        "topic": "AWS Elastic Beanstalk",
        "category": "Compute",
        "provider": "aws",
        "urls": [
            "https://docs.aws.amazon.com/elasticbeanstalk/latest/dg/Welcome.html",
        ],
        "search_queries": [
            "AWS Elastic Beanstalk deployment tutorial",
        ],
    },

    # 💾 Storage
    {
        "topic": "Amazon S3",
        "category": "Storage",
        "provider": "aws",
        "urls": [
            "https://docs.aws.amazon.com/AmazonS3/latest/userguide/Welcome.html",
        ],
        "search_queries": [
            "AWS S3 complete guide",
            "AWS S3 security best practices",
        ],
    },
    {
        "topic": "Amazon EFS",
        "category": "Storage",
        "provider": "aws",
        "urls": [
            "https://docs.aws.amazon.com/efs/latest/ug/whatisefs.html",
        ],
        "search_queries": [
            "AWS EFS file storage tutorial",
        ],
    },

    # 🗄️ Database
    {
        "topic": "Amazon RDS",
        "category": "Database",
        "provider": "aws",
        "urls": [
            "https://docs.aws.amazon.com/AmazonRDS/latest/UserGuide/Welcome.html",
        ],
        "search_queries": [
            "AWS RDS database tutorial",
        ],
    },
    {
        "topic": "Amazon DynamoDB",
        "category": "Database",
        "provider": "aws",
        "urls": [
            "https://docs.aws.amazon.com/amazondynamodb/latest/developerguide/Introduction.html",
        ],
        "search_queries": [
            "AWS DynamoDB NoSQL tutorial",
        ],
    },

    # 🌐 Networking
    {
        "topic": "Amazon VPC",
        "category": "Networking",
        "provider": "aws",
        "urls": [
            "https://docs.aws.amazon.com/vpc/latest/userguide/what-is-amazon-vpc.html",
        ],
        "search_queries": [
            "AWS VPC networking tutorial",
        ],
    },
    {
        "topic": "Amazon Route 53",
        "category": "Networking",
        "provider": "aws",
        "urls": [
            "https://docs.aws.amazon.com/Route53/latest/DeveloperGuide/Welcome.html",
        ],
        "search_queries": [
            "AWS Route 53 DNS tutorial",
        ],
    },

    # 📦 Containers
    {
        "topic": "Amazon ECS",
        "category": "Containers",
        "provider": "aws",
        "urls": [
            "https://docs.aws.amazon.com/AmazonECS/latest/developerguide/Welcome.html",
        ],
        "search_queries": [
            "AWS ECS containers tutorial",
        ],
    },
    {
        "topic": "Amazon EKS",
        "category": "Containers",
        "provider": "aws",
        "urls": [
            "https://docs.aws.amazon.com/eks/latest/userguide/what-is-eks.html",
        ],
        "search_queries": [
            "AWS EKS Kubernetes tutorial",
        ],
    },

    # 🔐 Security
    {
        "topic": "AWS IAM",
        "category": "Security",
        "provider": "aws",
        "urls": [
            "https://docs.aws.amazon.com/IAM/latest/UserGuide/introduction.html",
        ],
        "search_queries": [
            "AWS IAM security best practices",
        ],
    },

    # 🤖 AI/ML
    {
        "topic": "Amazon SageMaker",
        "category": "AI/ML",
        "provider": "aws",
        "urls": [
            "https://docs.aws.amazon.com/sagemaker/latest/dg/whatis.html",
        ],
        "search_queries": [
            "AWS SageMaker machine learning tutorial",
        ],
    },

    # ═════════════════════════════════════════════
    #  ☁️  AZURE — Microsoft Azure
    # ═════════════════════════════════════════════

    # 🖥️ Compute
    {
        "topic": "Azure Virtual Machines",
        "category": "Compute",
        "provider": "azure",
        "urls": [
            "https://learn.microsoft.com/en-us/azure/virtual-machines/overview",
        ],
        "search_queries": [
            "Azure Virtual Machines tutorial",
            "Azure VM sizes explained",
        ],
    },
    {
        "topic": "Azure Functions",
        "category": "Compute",
        "provider": "azure",
        "urls": [
            "https://learn.microsoft.com/en-us/azure/azure-functions/functions-overview",
        ],
        "search_queries": [
            "Azure Functions serverless tutorial",
        ],
    },
    {
        "topic": "Azure App Service",
        "category": "Compute",
        "provider": "azure",
        "urls": [
            "https://learn.microsoft.com/en-us/azure/app-service/overview",
        ],
        "search_queries": [
            "Azure App Service web app tutorial",
        ],
    },

    # 💾 Storage
    {
        "topic": "Azure Blob Storage",
        "category": "Storage",
        "provider": "azure",
        "urls": [
            "https://learn.microsoft.com/en-us/azure/storage/blobs/storage-blobs-introduction",
        ],
        "search_queries": [
            "Azure Blob Storage tutorial",
        ],
    },
    {
        "topic": "Azure Files",
        "category": "Storage",
        "provider": "azure",
        "urls": [
            "https://learn.microsoft.com/en-us/azure/storage/files/storage-files-introduction",
        ],
        "search_queries": [
            "Azure Files storage tutorial",
        ],
    },

    # 🗄️ Database
    {
        "topic": "Azure SQL Database",
        "category": "Database",
        "provider": "azure",
        "urls": [
            "https://learn.microsoft.com/en-us/azure/azure-sql/database/sql-database-paas-overview",
        ],
        "search_queries": [
            "Azure SQL Database tutorial",
        ],
    },
    {
        "topic": "Azure Cosmos DB",
        "category": "Database",
        "provider": "azure",
        "urls": [
            "https://learn.microsoft.com/en-us/azure/cosmos-db/introduction",
        ],
        "search_queries": [
            "Azure Cosmos DB NoSQL tutorial",
        ],
    },

    # 🌐 Networking
    {
        "topic": "Azure Virtual Network",
        "category": "Networking",
        "provider": "azure",
        "urls": [
            "https://learn.microsoft.com/en-us/azure/virtual-network/virtual-networks-overview",
        ],
        "search_queries": [
            "Azure Virtual Network tutorial",
        ],
    },
    {
        "topic": "Azure Load Balancer",
        "category": "Networking",
        "provider": "azure",
        "urls": [
            "https://learn.microsoft.com/en-us/azure/load-balancer/load-balancer-overview",
        ],
        "search_queries": [
            "Azure Load Balancer tutorial",
        ],
    },

    # 📦 Containers
    {
        "topic": "Azure Kubernetes Service (AKS)",
        "category": "Containers",
        "provider": "azure",
        "urls": [
            "https://learn.microsoft.com/en-us/azure/aks/what-is-aks",
        ],
        "search_queries": [
            "Azure AKS Kubernetes tutorial",
        ],
    },
    {
        "topic": "Azure Container Instances",
        "category": "Containers",
        "provider": "azure",
        "urls": [
            "https://learn.microsoft.com/en-us/azure/container-instances/container-instances-overview",
        ],
        "search_queries": [
            "Azure Container Instances tutorial",
        ],
    },

    # 🔐 Security
    {
        "topic": "Microsoft Entra ID",
        "category": "Security",
        "provider": "azure",
        "urls": [
            "https://learn.microsoft.com/en-us/entra/fundamentals/whatis",
        ],
        "search_queries": [
            "Microsoft Entra ID Azure AD tutorial",
        ],
    },

    # 🤖 AI/ML
    {
        "topic": "Azure OpenAI Service",
        "category": "AI/ML",
        "provider": "azure",
        "urls": [
            "https://learn.microsoft.com/en-us/azure/ai-services/openai/overview",
        ],
        "search_queries": [
            "Azure OpenAI Service GPT tutorial",
        ],
    },
    {
        "topic": "Azure Machine Learning",
        "category": "AI/ML",
        "provider": "azure",
        "urls": [
            "https://learn.microsoft.com/en-us/azure/machine-learning/overview-what-is-azure-machine-learning",
        ],
        "search_queries": [
            "Azure Machine Learning tutorial",
        ],
    },

    # ═════════════════════════════════════════════
    #  ☁️  GCP — Google Cloud Platform
    # ═════════════════════════════════════════════

    # 🖥️ Compute
    {
        "topic": "Google Compute Engine",
        "category": "Compute",
        "provider": "gcp",
        "urls": [
            "https://cloud.google.com/compute/docs/overview",
        ],
        "search_queries": [
            "Google Compute Engine tutorial",
        ],
    },
    {
        "topic": "Google Cloud Functions",
        "category": "Compute",
        "provider": "gcp",
        "urls": [
            "https://cloud.google.com/functions/docs/concepts/overview",
        ],
        "search_queries": [
            "Google Cloud Functions serverless tutorial",
        ],
    },
    {
        "topic": "Google Cloud Run",
        "category": "Compute",
        "provider": "gcp",
        "urls": [
            "https://cloud.google.com/run/docs/overview/what-is-cloud-run",
        ],
        "search_queries": [
            "Google Cloud Run containers tutorial",
        ],
    },

    # 💾 Storage
    {
        "topic": "Google Cloud Storage",
        "category": "Storage",
        "provider": "gcp",
        "urls": [
            "https://cloud.google.com/storage/docs/introduction",
        ],
        "search_queries": [
            "Google Cloud Storage tutorial",
        ],
    },
    {
        "topic": "Google Filestore",
        "category": "Storage",
        "provider": "gcp",
        "urls": [
            "https://cloud.google.com/filestore/docs/overview",
        ],
        "search_queries": [
            "Google Filestore NFS tutorial",
        ],
    },

    # 🗄️ Database
    {
        "topic": "Google Cloud SQL",
        "category": "Database",
        "provider": "gcp",
        "urls": [
            "https://cloud.google.com/sql/docs/introduction",
        ],
        "search_queries": [
            "Google Cloud SQL tutorial",
        ],
    },
    {
        "topic": "Google Cloud Spanner",
        "category": "Database",
        "provider": "gcp",
        "urls": [
            "https://cloud.google.com/spanner/docs/overview",
        ],
        "search_queries": [
            "Google Cloud Spanner tutorial",
        ],
    },
    {
        "topic": "Google Firestore",
        "category": "Database",
        "provider": "gcp",
        "urls": [
            "https://cloud.google.com/firestore/docs/overview",
        ],
        "search_queries": [
            "Google Firestore NoSQL tutorial",
        ],
    },

    # 🌐 Networking
    {
        "topic": "Google VPC",
        "category": "Networking",
        "provider": "gcp",
        "urls": [
            "https://cloud.google.com/vpc/docs/overview",
        ],
        "search_queries": [
            "Google Cloud VPC networking tutorial",
        ],
    },
    {
        "topic": "Google Cloud Load Balancing",
        "category": "Networking",
        "provider": "gcp",
        "urls": [
            "https://cloud.google.com/load-balancing/docs/load-balancing-overview",
        ],
        "search_queries": [
            "Google Cloud Load Balancing tutorial",
        ],
    },

    # 📦 Containers
    {
        "topic": "Google Kubernetes Engine (GKE)",
        "category": "Containers",
        "provider": "gcp",
        "urls": [
            "https://cloud.google.com/kubernetes-engine/docs/concepts/kubernetes-engine-overview",
        ],
        "search_queries": [
            "Google GKE Kubernetes tutorial",
        ],
    },

    # 🔐 Security
    {
        "topic": "Google Cloud IAM",
        "category": "Security",
        "provider": "gcp",
        "urls": [
            "https://cloud.google.com/iam/docs/overview",
        ],
        "search_queries": [
            "Google Cloud IAM tutorial",
        ],
    },

    # 🤖 AI/ML
    {
        "topic": "Google Vertex AI",
        "category": "AI/ML",
        "provider": "gcp",
        "urls": [
            "https://cloud.google.com/vertex-ai/docs/start/introduction-unified-platform",
        ],
        "search_queries": [
            "Google Vertex AI machine learning tutorial",
        ],
    },
]

# ─────────────────────────────────────────────
# Résumé
# ─────────────────────────────────────────────
total = len(COURSES)
print(f"✅ {total} cours configurés\n")

for provider in PROVIDERS:
    courses = [c for c in COURSES if c["provider"] == provider]
    print(f"☁️  {provider.upper()} — {len(courses)} cours")
    for cat in sorted(set(c["category"] for c in courses)):
        cat_courses = [c for c in courses if c["category"] == cat]
        for c in cat_courses:
            print(f"   📘 [{cat}] {c['topic']}")
    print()

✅ 40 cours configurés

☁️  AWS — 13 cours
   📘 [AI/ML] Amazon SageMaker
   📘 [Compute] Amazon EC2
   📘 [Compute] AWS Lambda
   📘 [Compute] AWS Elastic Beanstalk
   📘 [Containers] Amazon ECS
   📘 [Containers] Amazon EKS
   📘 [Database] Amazon RDS
   📘 [Database] Amazon DynamoDB
   📘 [Networking] Amazon VPC
   📘 [Networking] Amazon Route 53
   📘 [Security] AWS IAM
   📘 [Storage] Amazon S3
   📘 [Storage] Amazon EFS

☁️  AZURE — 14 cours
   📘 [AI/ML] Azure OpenAI Service
   📘 [AI/ML] Azure Machine Learning
   📘 [Compute] Azure Virtual Machines
   📘 [Compute] Azure Functions
   📘 [Compute] Azure App Service
   📘 [Containers] Azure Kubernetes Service (AKS)
   📘 [Containers] Azure Container Instances
   📘 [Database] Azure SQL Database
   📘 [Database] Azure Cosmos DB
   📘 [Networking] Azure Virtual Network
   📘 [Networking] Azure Load Balancer
   📘 [Security] Microsoft Entra ID
   📘 [Storage] Azure Blob Storage
   📘 [Storage] Azure Files

☁️  GCP — 13 cours
   📘 [AI/ML] Google Vertex AI
   📘 [

In [7]:
# Cellule 6 — 🤖 Classe IntelligentScraper
# ==========================================
# Le moteur principal :
#   1. Scrape le HTML brut depuis les URLs officielles
#   2. Enrichit avec GPT-4o-mini (simule un web search)
#   3. Formate en Markdown structuré avec GPT-4o-mini
#   4. Sauvegarde en .md + métadonnées JSON
#   5. Détecte les changements (hash MD5) pour le cron

# Sélecteurs CSS par provider
CONTENT_SELECTORS = {
    "aws":   {"content": "#main-col-body",            "title": "h1"},
    "azure": {"content": "main#main",                 "title": "h1"},
    "gcp":   {"content": "article.devsite-article",   "title": "h1"},
}


class IntelligentScraper:
    """
    Scraper IA qui génère des cours Markdown structurés
    à partir des documentations cloud officielles + GPT-4o-mini.
    """

    def __init__(self, output_dir: str, openai_client: AzureOpenAI, model: str):
        self.output_dir = output_dir
        self.client = openai_client
        self.model = model
        self.stats = {"success": 0, "failed": 0, "skipped": 0}
        self.results = []

    # ─────────────────────────────────────────
    # Utilitaires
    # ─────────────────────────────────────────

    @staticmethod
    def _slugify(topic: str) -> str:
        """Convertit un nom de topic en nom de fichier."""
        return re.sub(r'[^\w\s-]', '', topic).strip().replace(' ', '_').lower()

    @staticmethod
    def _hash_content(content: str) -> str:
        """Hash MD5 pour détecter les changements."""
        return hashlib.md5(content.encode("utf-8")).hexdigest()

    def _has_changed(self, filepath: str, new_hash: str) -> bool:
        """Vérifie si le contenu a changé depuis la dernière fois."""
        meta_path = filepath.replace(".md", "_meta.json")
        if not os.path.exists(meta_path):
            return True
        try:
            with open(meta_path, "r", encoding="utf-8") as f:
                meta = json.load(f)
            return meta.get("content_hash") != new_hash
        except Exception:
            return True

    # ─────────────────────────────────────────
    # Étape 1 : Scraper le HTML
    # ─────────────────────────────────────────

    async def _fetch_page(self, url: str) -> Optional[str]:
        """Télécharge une page web."""
        headers = {
            "User-Agent": (
                "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                "AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36"
            ),
            "Accept": "text/html,application/xhtml+xml",
            "Accept-Language": "en-US,en;q=0.9",
        }
        try:
            connector = aiohttp.TCPConnector(ssl=False)
            async with aiohttp.ClientSession(connector=connector) as session:
                async with session.get(
                    url, headers=headers, timeout=aiohttp.ClientTimeout(total=30)
                ) as resp:
                    if resp.status == 200:
                        return await resp.text()
                    logger.error(f"❌ HTTP {resp.status} → {url}")
                    return None
        except Exception as e:
            logger.error(f"❌ Fetch error {url} → {e}")
            return None

    def _extract_text(self, html: str, provider: str) -> str:
        """Extrait le texte brut du HTML."""
        soup = BeautifulSoup(html, "lxml")
        sel = CONTENT_SELECTORS.get(provider, {})
        content = soup.select_one(sel.get("content", "main"))
        if not content:
            content = soup.select_one("main") or soup.select_one("article") or soup.body
        if not content:
            return ""
        for tag in content.find_all(["script", "style", "nav", "footer", "header", "aside"]):
            tag.decompose()
        return content.get_text(separator="\n", strip=True)

    # ─────────────────────────────────────────
    # Étape 2 : Enrichir via GPT
    # ─────────────────────────────────────────

    def _enrich_with_gpt(self, query: str) -> str:
        """Utilise GPT-4o-mini pour enrichir le contenu."""
        try:
            resp = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "Tu es un expert cloud computing senior. "
                            "Donne des informations factuelles, techniques et détaillées. "
                            "Inclus : architecture, cas d'usage, pricing, sécurité, best practices."
                        ),
                    },
                    {
                        "role": "user",
                        "content": f"Donne-moi des informations détaillées et techniques sur : {query}",
                    },
                ],
                max_tokens=2000,
                temperature=0.3,
            )
            return resp.choices[0].message.content
        except Exception as e:
            logger.error(f"❌ GPT enrich error : {e}")
            return ""

    # ─────────────────────────────────────────
    # Étape 3 : Formater en Markdown avec GPT
    # ─────────────────────────────────────────

    def _format_to_markdown(self, course: dict, raw_content: str, enriched_content: str) -> str:
        """GPT-4o-mini transforme tout en Markdown structuré."""

        sources_list = "\n".join([f"- [{url}]({url})" for url in course["urls"]])

        prompt = f"""Tu es un expert en rédaction de documentation technique cloud.

MISSION : Crée un cours Markdown COMPLET, DÉTAILLÉ et PÉDAGOGIQUE sur "{course['topic']}" ({course['provider'].upper()}).

CONTENU BRUT SCRAPÉ DE LA DOCUMENTATION OFFICIELLE :
```
{raw_content[:10000]}
```

INFORMATIONS COMPLÉMENTAIRES (recherche) :
```
{enriched_content[:6000]}
```

STRUCTURE OBLIGATOIRE DU FICHIER MARKDOWN :

# {course['topic']}

> **Provider** : {course['provider'].upper()}
> **Catégorie** : {course['category']}
> **Dernière mise à jour** : {datetime.utcnow().strftime('%Y-%m-%d')}
> **Sources** :
{sources_list}

## 📋 Résumé
[Description claire et concise du service en 4-6 lignes]

## 🎯 Cas d'utilisation
[Liste des cas d'usage concrets avec exemples réels]

## 🏗️ Architecture et concepts clés
[Explication détaillée de l'architecture avec les composants principaux]
[Ajouter un schéma textuel si pertinent]

## 🚀 Guide de démarrage rapide
[Étapes concrètes pour commencer]
[Commandes CLI dans des blocs de code]
[Console web steps]

## ⚙️ Configuration et options
[Options de configuration importantes]
[Tableaux des paramètres si applicable]

## 💰 Modèle de tarification
[Explication du pricing]
[Tiers gratuits]
[Tableau comparatif des plans si applicable]

## 🔒 Sécurité et bonnes pratiques
[Best practices de sécurité]
[Configurations recommandées]
[Erreurs courantes à éviter]

## 🔄 Intégrations
[Services complémentaires]
[Comment s'intègre avec le reste de l'écosystème]

## 📊 Comparaison avec les alternatives
[Tableau comparatif : {course['topic']} vs équivalents sur les 2 autres clouds]
| Critère | AWS | Azure | GCP |
[Remplir le tableau]

## 📝 Exemples de code
[Minimum 2-3 exemples pratiques]
[CLI, SDK, ou IaC (Terraform/CloudFormation/ARM)]

## ❓ FAQ
[5-8 questions fréquentes avec réponses]

## 📚 Sources et références
[Liste complète de toutes les sources utilisées avec URLs]

RÈGLES :
- Sois TRÈS détaillé et pédagogique (minimum 2000 mots)
- Utilise des emojis pour les titres
- Ajoute des blocs de code avec le bon langage (bash, python, json, yaml, etc.)
- Ajoute des tableaux quand c'est pertinent
- Chaque section doit avoir du contenu substantiel
- Cite TOUJOURS les sources à la fin
- Le cours doit être compréhensible pour un débutant mais utile pour un expert

Génère le cours complet maintenant :"""

        try:
            resp = self.client.chat.completions.create(
                model=self.model,
                messages=[
                    {
                        "role": "system",
                        "content": (
                            "Tu es un rédacteur technique senior spécialisé en cloud computing. "
                            "Tu produis des cours Markdown parfaitement structurés, détaillés, "
                            "pédagogiques et professionnels. Tu cites toujours tes sources."
                        ),
                    },
                    {"role": "user", "content": prompt},
                ],
                max_tokens=4096,
                temperature=0.4,
            )
            return resp.choices[0].message.content
        except Exception as e:
            logger.error(f"❌ GPT format error : {e}")
            return ""

    # ─────────────────────────────────────────
    # Pipeline complet pour un cours
    # ─────────────────────────────────────────

    async def process_course(self, course: dict) -> Optional[str]:
        """Pipeline complet : scrape → enrichit → formate → sauvegarde."""
        topic = course["topic"]
        provider = course["provider"]
        filename = self._slugify(topic)
        output_dir = os.path.join(self.output_dir, provider)
        os.makedirs(output_dir, exist_ok=True)
        md_path = os.path.join(output_dir, f"{filename}.md")

        logger.info(f"[{provider.upper()}] 🔄 {topic}")

        # ── 1. Scraper les URLs ──
        raw_parts = []
        for url in course["urls"]:
            logger.info(f"[{provider.upper()}]    🌐 Scraping {url}")
            html = await self._fetch_page(url)
            if html:
                text = self._extract_text(html, provider)
                if text:
                    raw_parts.append(text)
                    logger.info(f"[{provider.upper()}]    ✅ {len(text):,} chars")
            await asyncio.sleep(1)

        raw_content = "\n\n".join(raw_parts)

        # ── 2. Enrichir via GPT ──
        enriched_parts = []
        for query in course.get("search_queries", []):
            logger.info(f"[{provider.upper()}]    🔍 Recherche : {query}")
            result = self._enrich_with_gpt(query)
            if result:
                enriched_parts.append(result)
            time.sleep(1)

        enriched_content = "\n\n".join(enriched_parts)

        # ── 3. Formater en Markdown ──
        logger.info(f"[{provider.upper()}]    🧠 Formatage Markdown...")
        markdown = self._format_to_markdown(course, raw_content, enriched_content)

        if not markdown or len(markdown) < 500:
            logger.error(f"[{provider.upper()}]    ❌ Contenu insuffisant pour {topic}")
            self.stats["failed"] += 1
            return None

        # ── 4. Vérifier si changement ──
        new_hash = self._hash_content(markdown)
        if not self._has_changed(md_path, new_hash):
            logger.info(f"[{provider.upper()}]    ⏭️ Inchangé : {topic}")
            self.stats["skipped"] += 1
            return None

        # ── 5. Sauvegarder .md ──
        with open(md_path, "w", encoding="utf-8") as f:
            f.write(markdown)

        # ── 6. Sauvegarder métadonnées JSON ──
        metadata = {
            "topic": topic,
            "category": course["category"],
            "provider": provider,
            "urls": course["urls"],
            "search_queries": course.get("search_queries", []),
            "content_hash": new_hash,
            "char_count": len(markdown),
            "scraped_at": datetime.utcnow().isoformat(),
            "md_file": md_path,
        }
        meta_path = os.path.join(output_dir, f"{filename}_meta.json")
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(metadata, f, ensure_ascii=False, indent=2)

        logger.info(f"[{provider.upper()}]    ✅ Sauvegardé : {md_path} ({len(markdown):,} chars)")

        self.stats["success"] += 1
        self.results.append({
            "provider": provider.upper(),
            "topic": topic,
            "category": course["category"],
            "chars": len(markdown),
            "file": md_path,
        })
        return md_path

    # ─────────────────────────────────────────
    # Traiter tous les cours
    # ─────────────────────────────────────────

    async def process_all(self, courses: list) -> dict:
        """Traite tous les cours séquentiellement."""
        start = time.time()

        logger.info("=" * 60)
        logger.info(f"🚀 Scraping intelligent — {len(courses)} cours")
        logger.info("=" * 60)

        for i, course in enumerate(courses):
            logger.info(f"\n{'─' * 50}")
            logger.info(f"📘 [{i+1}/{len(courses)}] {course['topic']} ({course['provider'].upper()})")
            logger.info(f"{'─' * 50}")

            await self.process_course(course)
            if i < len(courses) - 1:
                await asyncio.sleep(2)

        duration = time.time() - start
        logger.info(f"\n{'=' * 60}")
        logger.info(f"✅ TERMINÉ en {duration:.0f}s ({duration/60:.1f} min)")
        logger.info(f"{'=' * 60}")

        return {"stats": self.stats, "duration": duration}


print("✅ Classe IntelligentScraper chargée")

✅ Classe IntelligentScraper chargée


In [8]:
# Cellule 7 — 🚀 LANCER LE SCRAPING + 🔄 CRON
# ==========================================
# MODE :
#   - enabled: False → Scraping unique (test)
#   - enabled: True  → Cron automatique (production)

CRON_CONFIG = {
    "enabled": False,             # ← Mettre True pour activer le cron
    "interval_hours": 24,         # Intervalle entre chaque scraping
    "max_runs": 0,                # 0 = infini
    "stop_if_no_changes": False,  # True = arrêter si rien n'a changé
}

# ─────────────────────────────────────────
# 🎯 QUOI SCRAPER ? Décommenter UNE option
# ─────────────────────────────────────────

# Option 1 : UN SEUL cours (pour tester rapidement ~2 min)
courses_to_scrape = [COURSES[0]]

# Option 2 : UN SEUL provider (~10-15 min)
# courses_to_scrape = [c for c in COURSES if c["provider"] == "azure"]

# Option 3 : TOUT scraper (~30-45 min)
# courses_to_scrape = COURSES

# ─────────────────────────────────────────
OUTPUT_DIR = os.path.join(DATA_DIR, "raw")


async def run_scraping():
    """Exécute un cycle complet de scraping intelligent."""
    scraper = IntelligentScraper(
        output_dir=OUTPUT_DIR,
        openai_client=client,
        model=AZURE_OPENAI_CONFIG["model"],
    )

    result = await scraper.process_all(courses_to_scrape)

    # Afficher les résultats
    print("\n" + "=" * 60)
    print("📊 RÉSULTATS")
    print("=" * 60)
    print(f"   ✅ Nouveaux / Mis à jour : {scraper.stats['success']}")
    print(f"   ⏭️  Inchangés             : {scraper.stats['skipped']}")
    print(f"   ❌ Échecs                 : {scraper.stats['failed']}")
    print(f"   ⏱️  Durée                  : {result['duration']:.0f}s ({result['duration']/60:.1f} min)")

    # Sauvegarder le rapport
    report = {
        "timestamp": datetime.utcnow().isoformat(),
        "duration_seconds": round(result["duration"], 1),
        "stats": scraper.stats,
        "courses_scraped": scraper.results,
        "cron_enabled": CRON_CONFIG["enabled"],
    }
    report_name = f"ai_scraping_{datetime.utcnow().strftime('%Y%m%d_%H%M%S')}.json"
    report_path = os.path.join(DATA_DIR, "logs", report_name)
    with open(report_path, "w", encoding="utf-8") as f:
        json.dump(report, f, ensure_ascii=False, indent=2)
    print(f"   📄 Rapport : {report_name}")

    return scraper


async def cron_loop():
    """Boucle Cron : relance le scraping à intervalle régulier."""
    run_count = 0
    interval = CRON_CONFIG["interval_hours"] * 3600

    while True:
        run_count += 1
        print("\n" + "🔄" * 30)
        print(f"🕐 CRON RUN #{run_count} — {datetime.utcnow().strftime('%Y-%m-%d %H:%M:%S')} UTC")
        print("🔄" * 30)

        scraper = await run_scraping()
        has_changes = scraper.stats["success"] > 0

        if has_changes:
            print(f"\n📢 {scraper.stats['success']} cours mis à jour")
            print("   🔄 → Déclencher re-indexing RAG (Étape 2)")
            # TODO: Connecter avec le pipeline RAG
            # await trigger_rag_reindex()
        else:
            print("\n✅ Aucun changement. Base de connaissances à jour.")
            if CRON_CONFIG["stop_if_no_changes"]:
                print("⏹️ Arrêt du cron (aucun changement)")
                break

        if CRON_CONFIG["max_runs"] > 0 and run_count >= CRON_CONFIG["max_runs"]:
            print(f"\n⏹️ Max runs atteint ({CRON_CONFIG['max_runs']})")
            break

        next_run = datetime.utcnow() + timedelta(hours=CRON_CONFIG["interval_hours"])
        print(f"\n⏳ Prochain run : {next_run.strftime('%Y-%m-%d %H:%M:%S')} UTC")
        print(f"   (dans {CRON_CONFIG['interval_hours']}h)")
        print("   💡 Pour arrêter : bouton ⏹️ (Interrupt kernel)")

        await asyncio.sleep(interval)


# ─────────────────────────────────────────
# 🚀 LANCEMENT
# ─────────────────────────────────────────
print(f"📚 Cours à scraper : {len(courses_to_scrape)}")
print(f"🔄 Cron : {'ACTIVÉ' if CRON_CONFIG['enabled'] else 'DÉSACTIVÉ'}\n")

if CRON_CONFIG["enabled"]:
    print(f"   📅 Intervalle   : toutes les {CRON_CONFIG['interval_hours']}h")
    print(f"   🔁 Max runs     : {'∞' if CRON_CONFIG['max_runs'] == 0 else CRON_CONFIG['max_runs']}")
    print(f"   ⏹️  Stop si RAS : {CRON_CONFIG['stop_if_no_changes']}\n")
    await cron_loop()
else:
    await run_scraping()

2026-02-17 13:13:16,523 [INFO] [AWS]    ✅ 9,598 chars
2026-02-17 13:13:17,525 [INFO] [AWS]    🔍 Recherche : AWS EC2 instance types explained
2026-02-17 13:13:30,194 [INFO] HTTP Request: POST https://oai-subul-tuteur-dev.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-02-17 13:13:31,197 [INFO] [AWS]    🔍 Recherche : AWS EC2 best practices security
2026-02-17 13:13:40,988 [INFO] HTTP Request: POST https://oai-subul-tuteur-dev.openai.azure.com/openai/deployments/gpt-4o-mini/chat/completions?api-version=2024-12-01-preview "HTTP/1.1 200 OK"
2026-02-17 13:13:41,991 [INFO] [AWS]    🧠 Formatage Markdown...



📊 RÉSULTATS
   ✅ Nouveaux / Mis à jour : 1
   ⏭️  Inchangés             : 0
   ❌ Échecs                 : 0
   ⏱️  Durée                  : 51s (0.9 min)
   📄 Rapport : ai_scraping_20260217_131407.json


In [9]:
# Cellule 8 — 📊 Tableau des résultats
# ==========================================
if scraper.results:
    print("📋 COURS GÉNÉRÉS\n")
    print(f"  {'#':<4} {'Provider':<10} {'Catégorie':<12} {'Cours':<35} {'Taille':>10}")
    print("  " + "─" * 75)

    for i, r in enumerate(scraper.results):
        topic = r['topic'][:32] + "..." if len(r['topic']) > 35 else r['topic']
        print(f"  {i:<4} {r['provider']:<10} {r['category']:<12} {topic:<35} {r['chars']:>10,}")

    print(f"\n  {'─' * 75}")

    for provider in ["AWS", "AZURE", "GCP"]:
        docs = [r for r in scraper.results if r['provider'] == provider]
        if docs:
            total = sum(d['chars'] for d in docs)
            print(f"  ☁️  {provider:<8} → {len(docs)} cours | {total:,} caractères")

    total_all = sum(r['chars'] for r in scraper.results)
    print(f"\n  📈 TOTAL → {len(scraper.results)} cours | {total_all:,} caractères")
else:
    print("⚠️ Aucun cours généré")
    print("   → Vérifiez les logs ci-dessus")

NameError: name 'scraper' is not defined